# Bayesian estimation and regularization
Book: Pattern Matching, Penalized Likelihood; Statistical Learning Theory and Regularization, ridge.

Optional background: Alisa's math notes, [Bias & variance](https://alisawuffles.notion.site/math-notes#3737eb8736058073b5c2c112ba2cba13). Connect the decomposition to repeated fits in our perturbation experiment.
These selected readings are optional support. The classroom examples define the required scope.

Keep degree 12 fixed. The only model-selection variable is the ridge penalty.
The library minimizes sum of squared errors plus alpha times squared coefficients.
It scales features using training data and leaves the intercept unpenalized.

## Setup
Run this cell once. Helpers support the experiments below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 12})

def regression_data(seed=600, n=100):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2, 2, n)
    y = 1 + 2*x + 0.7*x*x + rng.normal(0, 1, n)
    return x.reshape(-1, 1), y

def regression_split():
    x, y = regression_data()
    order = np.random.default_rng(17).permutation(len(y))
    a, b = order[:70], order[70:]
    return x[a], x[b], y[a], y[b]

def polynomial_model(degree=3, alpha=0):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import PolynomialFeatures, StandardScaler
    from sklearn.linear_model import LinearRegression, Ridge
    estimator = LinearRegression() if alpha == 0 else Ridge(alpha=alpha)
    return make_pipeline(PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(), estimator)

## Bayesian estimation: adding a prior (6 minutes)
A Beta(2,2) prior favors probabilities near the middle over the extremes.
Predict how its influence differs for 8/10 and 80/100, then compare.

In [ ]:
from scipy.stats import beta
p = np.linspace(.001, .999, 999)
prior_a, prior_b = 2, 2
fig, ax = plt.subplots()
ax.plot(p, beta.pdf(p, prior_a, prior_b), label="Prior Beta(2,2)")
for successes, n in [(8, 10), (80, 100)]:
    a, b = prior_a+successes, prior_b+n-successes
    ax.plot(p, beta.pdf(p, a, b), label=f"Posterior after {successes}/{n}")
    print(successes, n, "MLE", successes/n, "posterior mean", a/(a+b))
ax.set(xlabel="Unknown probability p", ylabel="Probability density over p")
ax.legend()
plt.show()

Unlike relative likelihood, each displayed curve is a normalized density over p.
Change the prior to Beta(1,1). Which posterior mean moves more, and why?

## A. The cost of flexibility (30 minutes)
Predict training error, validation error, and coefficient size as alpha grows.

In [ ]:
from sklearn.metrics import mean_squared_error
xr, xv, yr, yv = regression_split()
alphas = [0., .01, 1., 100.]
grid = np.linspace(-2, 2, 300).reshape(-1, 1)
models, validation_mse = [], []
fig, ax = plt.subplots()
ax.scatter(xr[:, 0], yr, c="gray", alpha=.5)
for alpha in alphas:
    model = polynomial_model(12, alpha).fit(xr, yr)
    models.append(model)
    validation_mse.append(mean_squared_error(yv, model.predict(xv)))
    print(alpha, "train MSE", mean_squared_error(yr, model.predict(xr)),
          "validation MSE", validation_mse[-1],
          "coefficient norm", np.linalg.norm(model[-1].coef_))
    ax.plot(grid[:, 0], model.predict(grid), label=f"alpha={alpha}")
ax.set(xlabel="x", ylabel="Prediction", ylim=(-5, 12))
ax.legend()
plt.show()

Which preference does the penalty express? Why does feature scale matter?

Prediction:

Observation:

Explanation:

## B. Stability under new training noise (30 minutes)
Hold x fixed and perturb y slightly. Compare weak and moderate penalties.

In [ ]:
rng = np.random.default_rng(604)
perturbations = rng.normal(0, .3, (12, len(yr)))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, alpha in zip(axes, [0., 1.]):
    for perturbation in perturbations:
        changed_y = yr + perturbation
        model = polynomial_model(12, alpha).fit(xr, changed_y)
        ax.plot(grid[:, 0], model.predict(grid), alpha=.35)
    ax.set(title=f"alpha={alpha}", xlabel="x", ylim=(-5, 12))
axes[0].set_ylabel("Prediction")
plt.tight_layout()
plt.show()

These are repeated perturbations, not a posterior credible band.
Explain the stability change and any cost in fidelity to the training sample.

Prediction:

Observation:

Explanation:

## Final comparison, only after the choice is fixed
Set reveal_test=True once your choice and rationale are recorded.
Do not change alpha in response to this score.

In [ ]:
chosen = int(np.argmin(validation_mse))
reveal_test = False
if reveal_test:
    xt, yt = regression_data(seed=900, n=1000)
    print("Chosen alpha:", alphas[chosen])
    print("Test MSE:", mean_squared_error(yt, models[chosen].predict(xt)))

## Individual exit
With an unpenalized intercept, what predictor remains as alpha becomes huge?
Why would selecting alpha using training error undermine this experiment?